<a href="https://colab.research.google.com/github/BalasA90/-ml-product-reviews-project/blob/main/notebook/product_reviews_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Loading and inspecting the dataset

Before diving into analysis,we first need to load the dataset and take a look
at its structure.

In this step,we will:
- Load csv file from GitHub
- Check how many rows and columns we have
- Display the first few rows
- Review data types and basis metadata for each column

This will help types us ensure the dataset is correctly loaded an ready for further exploration.

In [ ]:
import pandas as pd

# load dataset from GitHub
url = "https://raw.githubusercontent.com//BalasA90/-ml-product-reviews-project/main/data/product_reviews_full.csv"
df = pd.read_csv(url)

# Print shape (number of rows and columns)
print("Dataset shape (rows and columns):",df.shape)

# Show first 5 rows
print("\nFirst 5 rows:")
display(df.head())

# Show column data types and non-null counts
print("\nDataset info:")
df.info()

## Checking for missing values

Missing data can cause problems during model trainind or analysis
Here,we will:
- Count the number of missing (NaN) values per column
- Visualize missing values using a heatmap

This will help us identify any columns that require cleaning or imputation.

In [ ]:
# Count missing values per column
print("Missing values per column:")
print(df.isna().sum())

In [ ]:
# Visualize missing data with seaborn heatmap
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
sns.heatmap(df.isna(), cbar=False, cmap="YlOrRd")
plt.title("Missing Values Heatmap")
plt.show()

## Sentiment analysis

This helps us:
- Understand the balance between classes
- Detect if the dataset is skewed

In [ ]:
# Count occurrences of each sentiment label
sentiment_counts = df['sentiment'].value_counts()

# Print counts
print("Sentiment distribution (counts):")
print(sentiment_counts)

In [ ]:
# Plot sentiment distribution
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(sentiment_counts.index, sentiment_counts.values, color=['skyblue', 'salmon'])
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.show()

In [ ]:
#Mini activitate: Ce se întâmplă cu prețurile?
#În setul nostru de date există o coloană product_price și am observat deja că tipul ei nu este numeric (float sau int), ci textual (object).
#Cercetează de ce este așa și ce tipuri de valori apar în această coloană. Apoi scrie cod pentru analiza de cercetare a datelor din coloana product_price.
#Ce trebuie să faci:
#Afișează tipul de date al coloanei product_price.
#Afișează câteva dintre primele valori.
#Afișează cele mai frecvente valori.
#Verifică dacă în această coloană există înregistrări textuale nestructurate, cum ar fi „Free” sau „Not Available”.
#Găsește și afișează toate valorile „nenumerice” diferite.
#Să vedem soluția.

# 1. Check the data type of the 'product_price' column
print("Data type of product_price column:", df['product_price'].dtype)

# 2. Display the first 10 values from the column
print("\nFirst 10 values in product_price column:")
print(df['product_price'].head(10))

# 3. Show the 20 most frequent values in the column
print("\nTop 20 most frequent values in product_price column:")
print(df['product_price'].value_counts().head(20))

# 4. Check for known non-numeric text values
problematic_values = ['Free', 'Not Available', 'N/A', 'None', '-', 'free', 'unknown', 'Unavailable']
# Identify rows containing these specific non-numeric values
mask_problematic = df['product_price'].astype(str).str.strip().isin(problematic_values)
df_problematic = df[mask_problematic]
print(f"\nFound {len(df_problematic)} rows with problematic textual values:")
display(df_problematic[['product_price']].drop_duplicates())

# 5. Find and display some of the non-numeric values
price_clean = df['product_price'].astype(str).str.strip()
price_numeric = pd.to_numeric(price_clean, errors='coerce')

invalid_prices = df[price_numeric.isna()]
print("Number of non-numeric prices:", len(invalid_prices))
display(invalid_prices[['product_price']].drop_duplicates().head(20))

In [ ]:
# Drop all rows with missing values
df = df.dropna()

# Display new shape of the dataset
print("New dataset shape:", df.shape)

# Count missing values per column
print("Missing values per column:")
print(df.isna().sum())

In [ ]:
#Mini activitate: Câte date am „pierdut”?
#Inainte să ștergi definitiv rândurile cu valori lipsă, calculează:
#Câte rânduri are setul de date înainte de ștergere?
#Câte rânduri rămân după dropna()?
#Care este procentajul pierderii de date (cât % din date s-a „pierdut”)?
#Bonus: Uită-te aleatoriu la câteva rânduri care urmează să fie șterse. Este ceva „interesant” printre ele? Vrei să păstrezi sau să completezi unele dintre aceste date?
#Gândește-te:

#Este setul de date încă suficient de mare pentru analiză?
#Este ștergerea cea mai simplă și mai potrivită opțiune pentru cazul nostru?
#Să vedem soluția.

import pandas as pd

# load dataset from GitHub
url = "https://raw.githubusercontent.com/vladimir-dresevic/ml-product-reviews-project/main/data/product_reviews_full.csv"

df = pd.read_csv(url)

# Count the number of rows before removing missing values
rows_before = len(df)

# Filter out rows that contain at least one missing value
rows_with_nan = df[df.isnull().any(axis=1)]

# Display a random sample of rows that will be removed
print("Randomly selected rows containing missing values:\n")
print(rows_with_nan.sample(n=min(5, len(rows_with_nan)), random_state=42))

# Remove rows with any missing values
df_cleaned = df.dropna()

# Count the number of rows after removing missing values
rows_after = len(df_cleaned)

# Show removal statistics
print("\n Removal statistics:")
print(f"- Number of rows before: {rows_before}")
print(f"- Number of rows after: {rows_after}")
print(f"- Number of removed rows: {rows_before - rows_after}")

In [ ]:
# Step 1: Convert to string and remove the 'USD' prefix and any leading/trailing spaces
df['product_price_cleaned'] = (
    df['product_price']
    .astype(str)
    .str.replace(r'$', '', regex=True)    # Remove '$'
    .str.replace(r'[^\d.]', '', regex=True) # Remove all non-numeric characters except the dot
    .str.strip()
)



# Step 2: Convert cleaned string to float
df['product_price'] = pd.to_numeric(df['product_price_cleaned'], errors='coerce')


# Step 3: Drop the temporary column
df = df.drop(columns=['product_price_cleaned'])


# Step 4: Drop any rows where conversion failed (still NaN)
df = df.dropna(subset=['product_price'])


# Step 5: Confirm result
print("Column type after parsing:", df['product_price'].dtype)
print("\nPrice summary:")
print(df['product_price'].describe())

In [ ]:
# Drop columns that are not useful for modeling
df = df.drop(columns=['review_uuid', 'product_name'])

# Preview remaining columns
print("Remaining columns:")
print(df.columns.tolist())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Show summary statistics grouped by sentiment
print("Price summary by sentiment:")
print(df.groupby('sentiment', observed=False)['product_price'].describe())

# Boxplot of prices by sentiment
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='sentiment', y='product_price')
plt.title("Product Price Distribution by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Product Price")
plt.grid(True)
plt.show()

## Lungimea textului

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create new column with length of each review_text
df['review_length'] = df['review_text'].astype(str).str.len()

# Show basic stats
print("Review length summary:")
print(df['review_length'].describe())

# Group by sentiment and describe review length
print("Review length statistics by sentiment:")
print(df.groupby('sentiment', observed=False)['review_length'].describe())

# Visualize distribution of review length by sentiment
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='sentiment', y='review_length')
plt.title("Review Length by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Review Length (number of characters)")
plt.grid(True)
plt.show()

## Train

In [ ]:
# Features and label
df = df.dropna(subset= ["review_title" ,"review_text","review_length","sentiment"])
x = df[["review_title", "review_text", "review_length"]]
y = df["sentiment"]


from sklearn.model_selection import train_test_split

# Train-test split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

## Vectorizare

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("title_tfidf", TfidfVectorizer(stop_words="english"), "review_title"),
        ("text_tfidf", TfidfVectorizer(stop_words="english"), "review_text"),
        ("length_scaler", MinMaxScaler(), ["review_length"])
    ]
)

## Antrenare si Evaluare

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# List of classifiers
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Support Vector Machine": LinearSVC()
}

# Train and evaluate
for name, model in models.items():
    print(f"\n{name}")
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)
    print(classification_report(y_test, y_pred))


Logistic Regression
              precision    recall  f1-score   support

    Negative       0.66      0.14      0.23      2474
     Neutral       0.30      0.01      0.02       638
    Positive       0.10      0.00      0.00      4461
    negative       0.65      0.86      0.74      6459
     neutral       0.55      0.51      0.53      2097
    positive       0.75      0.97      0.84     17063

    accuracy                           0.71     33192
   macro avg       0.50      0.41      0.39     33192
weighted avg       0.61      0.71      0.63     33192


Naive Bayes


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

    Negative       0.77      0.03      0.05      2474
     Neutral       0.00      0.00      0.00       638
    Positive       0.29      0.00      0.00      4461
    negative       0.62      0.85      0.72      6459
     neutral       0.64      0.26      0.37      2097
    positive       0.71      0.98      0.83     17063

    accuracy                           0.69     33192
   macro avg       0.51      0.35      0.33     33192
weighted avg       0.62      0.69      0.59     33192


Decision Tree
              precision    recall  f1-score   support

    Negative       0.29      0.25      0.27      2474
     Neutral       0.13      0.08      0.09       638
    Positive       0.19      0.11      0.14      4461
    negative       0.64      0.69      0.66      6459
     neutral       0.49      0.50      0.49      2097
    positive       0.76      0.85      0.80     17063

    accuracy                           0.64     33192
   macro